# Batch-of-One Penalty — Smoke Test

Goal of this notebook: confirm on Colab hardware that the per-row inference cost of a
GBDT collapses as batch size grows, and that most of the cost at batch size 1 comes from
the Python serving API rather than from tree traversal.

This is the smoke test only. It runs in roughly five minutes. Nothing here goes in the
paper. It exists to prove the harness works and the effect is real on the machine you
will actually use.

Run every cell top to bottom. Runtime type: CPU is correct, do not pick a GPU.

## 1. Install Libraries

Takes two to three minutes. Ignore any dependency warnings.

In [ ]:
!pip install -q lightgbm==4.7.0 xgboost==3.4.1 catboost==1.2.10 \
                onnxruntime==1.24.4 onnxmltools skl2onnx 2>&1 | tail -3
print("done")

## 2. Pin Threads to One

Thread count has to be fixed before any library is imported, otherwise the numbers move
between runs. This cell must run before the imports below it.

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"]:
    os.environ[_v] = "1"

import time, json, platform, statistics, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import onnxruntime as ort
warnings.filterwarnings("ignore")

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__,
      "| onnxruntime", ort.__version__)

## 3. Record the Machine

Colab hands out different CPUs on different days. Every latency number is meaningless
without knowing which machine produced it, so this gets written next to the results.

In [ ]:
def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return "unknown"

ENV = {
    "cpu": cpu_model(),
    "cores": os.cpu_count(),
    "python": platform.python_version(),
    "lightgbm": lgb.__version__,
    "xgboost": xgb.__version__,
    "onnxruntime": ort.__version__,
}
for k, v in ENV.items():
    print(f"{k:14s} {v}")

## 4. Mount Google Drive

The smoke test is short enough to survive a session, but the full experiment will not be,
so the saving path gets tested here. A folder called `batch_penalty` is created in the top
level of your Drive.

When you run this cell a popup asks you to pick your Google account and press Allow.

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/batch_penalty"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not available, saving locally instead:", e)
    SAVE_DIR = "/content/batch_penalty"

os.makedirs(SAVE_DIR, exist_ok=True)
with open(os.path.join(SAVE_DIR, "environment_smoke.json"), "w") as f:
    json.dump(ENV, f, indent=2)
print("results will be written to:", SAVE_DIR)

## 5. Load the Data

Online Shoppers Purchasing Intention from the UCI repository. Around 12,300 sessions on an
online store, and the label is whether the session ended in a purchase. It is a real
e-commerce decision task, which is what anchors the paper.

If the download fails the cell falls back to synthetic data so the smoke test still runs.

In [ ]:
URL = ("https://archive.ics.uci.edu/static/public/468/"
       "online+shoppers+purchasing+intention+dataset.zip")

def load_data():
    try:
        df = pd.read_csv(URL, compression="zip")
        y = df["Revenue"].astype(int).values
        Xdf = df.drop(columns=["Revenue"])
        for c in Xdf.columns:
            if Xdf[c].dtype == object or str(Xdf[c].dtype) == "bool":
                Xdf[c] = pd.factorize(Xdf[c])[0]
        return Xdf.values.astype(np.float32), y, "online_shoppers"
    except Exception as e:
        print("download failed, using synthetic:", type(e).__name__)
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=12000, n_features=17,
                                   n_informative=10, weights=[0.85],
                                   random_state=42)
        return X.astype(np.float32), y, "synthetic"

X, y, DATASET = load_data()
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test = np.ascontiguousarray(X[split:])
print(f"dataset={DATASET}  train={X_train.shape}  test={X_test.shape}  "
      f"positive rate={y.mean():.3f}")

## 6. Train the Three Models

Two hundred trees, depth six, single thread. These settings are held fixed so the
comparison is across serving paths and batch sizes, not across model capacity.

In [ ]:
N_TREES, DEPTH, SEED = 200, 6, 42

t0 = time.time()
lgbm = lgb.LGBMClassifier(n_estimators=N_TREES, max_depth=DEPTH, num_leaves=31,
                          n_jobs=1, verbose=-1, random_state=SEED).fit(X_train, y_train)
xgbm = xgb.XGBClassifier(n_estimators=N_TREES, max_depth=DEPTH, n_jobs=1,
                         tree_method="hist", random_state=SEED).fit(X_train, y_train)
cbm  = CatBoostClassifier(iterations=N_TREES, depth=DEPTH, thread_count=1,
                          verbose=0, random_seed=SEED).fit(X_train, y_train)
print(f"trained all three in {time.time()-t0:.1f}s")

## 7. Build the Serving Paths

This is the part that matters. The same trained model is called through several different
interfaces. The sklearn wrapper is what almost everyone writes in production. The native
booster call skips the wrapper's input checking. ONNX Runtime skips the training library
altogether.

In [ ]:
lgb_booster = lgbm.booster_
xgb_booster = xgbm.get_booster()

runners = {
    "lgb_sklearn": lambda b: lgbm.predict_proba(b),
    "lgb_booster": lambda b: lgb_booster.predict(b),
    "xgb_sklearn": lambda b: xgbm.predict_proba(b),
    "xgb_dmatrix": lambda b: xgb_booster.predict(xgb.DMatrix(b)),
    "xgb_inplace": lambda b: xgb_booster.inplace_predict(b),
    "cat_sklearn": lambda b: cbm.predict_proba(b),
}

# ONNX paths. Each converter takes slightly different arguments, so each is wrapped
# on its own and a failure removes only that one path.
from onnxmltools.convert import convert_lightgbm, convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

so = ort.SessionOptions()
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
init_types = [("input", FloatTensorType([None, X.shape[1]]))]

def register(name, sess):
    iname = sess.get_inputs()[0].name
    oname = sess.get_outputs()[-1].name          # probabilities, not the label
    runners[name] = lambda b, s=sess, i=iname, o=oname: s.run([o], {i: b})

def session(onx):
    return ort.InferenceSession(onx.SerializeToString(), so,
                                providers=["CPUExecutionProvider"])

try:
    register("lgb_onnx", session(convert_lightgbm(
        lgb_booster, initial_types=init_types, zipmap=False)))
    print("built lgb_onnx")
except Exception as e:
    print("skipped lgb_onnx:", type(e).__name__, str(e)[:130])

try:
    # the xgboost converter has no zipmap argument
    register("xgb_onnx", session(convert_xgboost(
        xgb_booster, initial_types=init_types)))
    print("built xgb_onnx")
except Exception as e:
    print("skipped xgb_onnx:", type(e).__name__, str(e)[:130])

try:
    # catboost exports to onnx itself, no external converter needed
    _cb_onnx = os.path.join(SAVE_DIR, "catboost_smoke.onnx")
    cbm.save_model(_cb_onnx, format="onnx")
    register("cat_onnx", ort.InferenceSession(
        _cb_onnx, so, providers=["CPUExecutionProvider"]))
    print("built cat_onnx")
except Exception as e:
    print("skipped cat_onnx:", type(e).__name__, str(e)[:130])

print("\nserving paths ready:", len(runners))
print(list(runners))

## 8. Correctness Check

Before timing anything, confirm the faster paths return the same predictions as the
sklearn call for the same library. A fast path that gives different answers is not a fast
path, it is a bug. This also protects the paper, because a reviewer will ask.

In [ ]:
probe = X_test[:64]

def to_prob(out):
    """Pull the positive-class probability out of whatever shape a path returns."""
    if isinstance(out, list):
        out = out[0]
    if isinstance(out, list) and isinstance(out[0], dict):      # catboost onnx zipmap
        return np.array([o[max(o)] for o in out], dtype=np.float64)
    arr = np.asarray(out)
    if arr.dtype == object:
        return np.array([o[max(o)] for o in arr], dtype=np.float64)
    arr = arr.reshape(len(probe), -1)
    return arr[:, -1].astype(np.float64)

groups = {"lightgbm": ["lgb_sklearn", "lgb_booster", "lgb_onnx"],
          "xgboost":  ["xgb_sklearn", "xgb_dmatrix", "xgb_inplace", "xgb_onnx"],
          "catboost": ["cat_sklearn", "cat_onnx"]}

for lib, paths in groups.items():
    paths = [p for p in paths if p in runners]
    ref = to_prob(runners[paths[0]](probe))
    print(f"\n{lib}  (reference: {paths[0]})")
    for p in paths[1:]:
        d = np.max(np.abs(to_prob(runners[p](probe)) - ref))
        flag = "ok" if d < 1e-4 else "CHECK THIS"
        print(f"   {p:14s} max abs diff {d:.2e}   {flag}")

## 9. The Timing Harness

For a given batch size the harness pushes a fixed number of rows through the model in
chunks of that size, times the whole pass, and divides by the row count. That keeps the
amount of work constant across batch sizes so the comparison is fair.

Warm-up passes are discarded because the first call pays one-off allocation costs. The
median across repeats is reported, since Colab machines are shared and the mean gets
dragged around by whatever else is running on the host.

In [ ]:
def per_row_latency(fn, X_pool, batch, workload=4096, repeats=7, warmup=2):
    rows = max(workload - workload % batch, batch)
    reps = rows // len(X_pool) + 1
    pool = np.ascontiguousarray(np.tile(X_pool, (reps, 1))[:rows])
    chunks = [pool[i:i + batch] for i in range(0, rows, batch)]

    for _ in range(warmup):
        for c in chunks[:min(8, len(chunks))]:
            fn(c)

    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        for c in chunks:
            fn(c)
        samples.append((time.perf_counter_ns() - t0) / rows / 1000.0)   # us per row
    return samples

## 10. Run the Smoke Grid

Seven batch sizes across every serving path. Results are written to Drive as they arrive,
and the cell skips any measurement already present, so re-running after a disconnect
picks up where it stopped.

In [ ]:
BATCHES = [1, 4, 16, 64, 256, 1024, 4096]
CSV = os.path.join(SAVE_DIR, "smoke_results.csv")

done = set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV)
    done = set(zip(prev["path"], prev["batch"]))
    print(f"resuming, {len(done)} measurements already on disk")

rows_out = []
for name, fn in runners.items():
    for b in BATCHES:
        if (name, b) in done:
            continue
        s = per_row_latency(fn, X_test, b)
        rec = {"dataset": DATASET, "path": name, "batch": b,
               "median_us": statistics.median(s),
               "min_us": min(s), "max_us": max(s),
               "iqr_us": np.subtract(*np.percentile(s, [75, 25]))}
        rows_out.append(rec)
        pd.DataFrame(rows_out).to_csv(
            CSV, mode="a", header=not os.path.exists(CSV), index=False)
        rows_out = []
        print(f"{name:14s} batch={b:6d}  {rec['median_us']:9.2f} us/row")

res = pd.read_csv(CSV).drop_duplicates(subset=["path", "batch"], keep="last")
print("\nsaved to", CSV)

## 11. The Table

Per-row latency in microseconds, and the penalty ratio between batch 1 and batch 4096.

In [ ]:
piv = res.pivot(index="path", columns="batch", values="median_us")
piv["penalty"] = piv[BATCHES[0]] / piv[BATCHES[-1]]
display(piv.round(2))

print("\nHow much of the batch-1 cost is the Python API rather than inference:")
for lib, wrapper, native in [("lightgbm", "lgb_sklearn", "lgb_booster"),
                             ("xgboost", "xgb_sklearn", "xgb_inplace")]:
    if wrapper in piv.index and native in piv.index:
        w, n = piv.loc[wrapper, 1], piv.loc[native, 1]
        print(f"  {lib:9s} wrapper {w:8.1f} us   native {n:8.1f} us   "
              f"overhead share {100*(w-n)/w:5.1f}%")

## 12. The Figure

Log-log curve of per-row cost against batch size. This is the shape the paper is about.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for name in piv.index:
    ax.plot(BATCHES, piv.loc[name, BATCHES].values, marker="o", label=name)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Batch size (rows per call)")
ax.set_ylabel("Latency per row (microseconds)")
ax.set_title(f"Per-row inference cost against batch size ({DATASET})")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "smoke_curve.png"), dpi=150)
plt.show()
print("figure saved to", os.path.join(SAVE_DIR, "smoke_curve.png"))

## What to Check Before Moving On

1. Every curve slopes downward and flattens out. If a curve is flat from the start, thread
   pinning did not take effect and the notebook needs restarting from cell one.
2. The native and ONNX paths are much cheaper than the sklearn wrapper at batch size 1.
3. The spread between `min_us` and `max_us` at batch 1 is small relative to the gap between
   paths. If the noise is the same size as the effect, the repeat count has to go up before
   the real experiment.

Send me the table from cell 11 and the figure, and I will build the full experiment
notebook on top of this harness.